In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

RESULT_CSV = None  # e.g. PLL_DIR / "pll_test_vdd_S5_20261022_100000.csv"; None = newest run
if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_test_vdd_*.csv")
                         if not p.stem.endswith(("_traces", "_spectra"))), key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
FIG_DIR = RESULT_CSV.with_name(RESULT_CSV.stem + "_figures")  # every figure as HTML + PNG
FIG_DIR.mkdir(exist_ok=True)
CODES = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]
VDD_STEP = 0.025  # [V] the measured supply is rounded to this grid to name the set voltage


def setting_col(df):
    """Name of the setting of every row of df: VCO codes and f_vco."""
    return ("c" + df["vco_tune_coarse"].astype(int).astype(str)
            + " min " + df["vco_current_min"].astype(int).astype(str)
            + " max " + df["vco_current_max"].astype(int).astype(str) + ", "
            + (df["f_vco"] / 1e6).round().astype(int).astype(str) + " MHz")


runs = pd.read_csv(RESULT_CSV)
runs["vdd"] = (runs["pll_vdd"] / VDD_STEP).round() * VDD_STEP
runs["setting"] = setting_col(runs)
runs = runs.drop_duplicates(["setting", "vdd"])  # the nominal supply is measured first and last: keep the first
traces = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv"))
spectra = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_spectra.csv"))
spectra["setting"] = setting_col(spectra)
# the nominal supply is measured first and last: keep the first
spectra = spectra[spectra["time"] == spectra.groupby(["setting", "vdd_set"])["time"].transform("min")]
ok = runs[runs["status"] == "ok"]
print(f"{RESULT_CSV.name}: {len(runs)} supply x setting, {len(ok)} measured ok; "
      f"settings {', '.join(runs['setting'].unique())}")

INK, GRID = "#52514e", "#e4e3df"
F_start, F_stop = 1e3, 20e6  # offset range of the phase-noise plots [Hz]
N_BINS = 300
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz"], gridcolor=GRID,
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"), range=[np.log10(F_start), np.log10(F_stop)])


def vdd_color(v):
    return sample_colorscale("Viridis", [min(max((v - 0.60) / 0.25, 0), 1) * 0.9])[0]


def binned(df):
    """Curve df (freq, level_dbm) in N_BINS log-spaced bins over F_start-F_stop (linear power mean, display only)."""
    df = df[(df["freq"] >= F_start) & (df["freq"] <= F_stop)].sort_values("freq")
    if len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(F_start, F_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def trace_of(r, kind):
    """The traces of run row r (same f_vco, codes, filter and measured supply) of one kind."""
    sel = (traces["kind"] == kind) & (traces["f_vco"] == r["f_vco"]) & (traces["pll_vdd"] == r["pll_vdd"])
    for c in CODES + ["set_current", "set_c1", "set_c2", "set_r1"]:
        sel &= traces[c] == r[c]
    return traces[sel]


def save(fig, name, width=1200):
    """HTML and PNG (kaleido) in FIG_DIR, then show the figure."""
    fig.update_layout(template="plotly_white", font=dict(size=16))
    fig.write_html(FIG_DIR / f"{name}.html")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", width=width, height=fig.layout.height or 600, scale=2)
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"PNG export failed: {e}")
    fig.show()


def settings_text(r):
    """Fixed setting of row r in one line: VCO codes, f_vco, output divider / pad, reference, loop filter."""
    return (f"VCO coarse {r['vco_tune_coarse']:.0f} min {r['vco_current_min']:.0f} max {r['vco_current_max']:.0f}, "
            f"f_vco {r['f_vco'] / 1e6:.0f} MHz, output /{r['total_div']:.0f} "
            f"(pad {r['f_vco'] / r['total_div'] / 1e6:.4g} MHz), f_ref {r['f_ref'] / 1e6:.4g} MHz, "
            f"loop filter Icp {r['set_current']:.0f} C1 {r['set_c1']:.0f} "
            f"C2 {r['set_c2']:.0f} R1 {r['set_r1']:.0f}, Vctrl pad off, core in reset")


def print_settings(figures, r, varied):
    """Print for the slides what `figures` show: the run, the fixed setting of row r, what varies."""
    print(f"{figures}\n  run: {RESULT_CSV.name}\n  fixed: {settings_text(r)}\n  varied: {varied}\n")


pd.DataFrame({
    "setting": runs["setting"],
    "PLL VDD [V]": runs["pll_vdd"],
    "PLL I [mA]": runs["pll_i"] * 1e3,
    "locked": runs["locked"],
    "carrier [dBm]": runs["carrier_dbm"],
    "jitter curve [ps]": runs["time_jitter_trace_ps"],
    "jitter noise [ps]": runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": runs["time_jitter_spur_ps"],
    "jitter Phase Noise mode [ps]": runs["time_jitter_pnm_ps"],
    "band Phase Noise mode [Hz]": runs["pnm_band"],
    "status": runs["status"],
}).round(3)

pll_test_vdd_S5_20261026_155550.csv: 2 supply x setting, 2 measured ok; settings c9 min 9 max 0, 760 MHz, c9 min 14 max 0, 1000 MHz


,setting,PLL VDD [V],PLL I [mA],locked,carrier [dBm],jitter curve [ps],jitter noise [ps],jitter spurs [ps],jitter Phase Noise mode [ps],band Phase Noise mode [Hz],status
0,"c9 min 9 max 0, 760 MHz",NaN,NaN,1,0.521,185.623,165.302,84.447,198.657,1000-2e+07,ok
1,"c9 min 14 max 0, 1000 MHz",NaN,NaN,1,1.582,22.731,18.066,13.796,20.229,1000-2e+07,ok


In [11]:
# RMS jitter on the pad against the PLL supply, one curve per setting: the full curve (total, noise part, spur part)
# and the analyzer's Phase Noise mode. One figure per quantity; plus the PLL supply current.
QUANTITIES = (("time_jitter_trace_ps", "Total RMS jitter (full curve)", "RMS jitter [ps]", "jitter_total_vs_vdd"),
              ("time_jitter_noise_ps", "RMS jitter of the noise (spurs removed)", "RMS jitter [ps]",
               "jitter_noise_vs_vdd"),
              ("time_jitter_spur_ps", "RMS jitter of the spurs", "RMS jitter [ps]", "jitter_spurs_vs_vdd"),
              ("time_jitter_pnm_ps", "RMS jitter (analyzer Phase Noise mode)", "RMS jitter [ps]", "jitter_pnm_vs_vdd"),
              ("pll_i", "PLL supply current", "current [mA]", "current_vs_vdd"))
for col, what, ytitle, name in QUANTITIES:
    fig = go.Figure()
    for setting, d in ok.groupby("setting", sort=False):
        d = d.sort_values("vdd")
        fig.add_scatter(x=d["vdd"], y=d[col] * (1e3 if col == "pll_i" else 1), mode="lines+markers", name=setting,
                        marker=dict(size=9), hovertemplate="%{x:.3f} V: %{y:.1f}<extra>" + setting + "</extra>")
    fig.update_layout(title=f"{what} vs PLL supply", height=500, legend=dict(orientation="h", y=-0.25))
    fig.update_xaxes(title="PLL supply [V]", gridcolor=GRID)
    fig.update_yaxes(title=ytitle, gridcolor=GRID, rangemode="tozero")
    save(fig, name)
for setting, d in ok.groupby("setting", sort=False):
    print_settings(", ".join(q[3] for q in QUANTITIES) + f" (PLL locked; curve: {setting})", d.iloc[0],
                   "PLL supply " + ", ".join(f"{v:.3f}" for v in sorted(d["vdd"])) + " V (E36312A CH2); "
                   "jitter on the pad "
                   f"over {d['common_band'].iloc[0]} Hz (full curve) and {d['pnm_band'].iloc[0]} Hz (Phase Noise mode)")

jitter_total_vs_vdd, jitter_noise_vs_vdd, jitter_spurs_vs_vdd, jitter_pnm_vs_vdd, current_vs_vdd (PLL locked; curve: c9 min 9 max 0, 760 MHz)
  run: pll_test_vdd_S5_20261026_155550.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8 (pad 95 MHz), f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7, Vctrl pad off, core in reset
  varied: PLL supply nan V (E36312A CH2); jitter on the pad over 1000-1e+07 Hz (full curve) and 1000-2e+07 Hz (Phase Noise mode)

jitter_total_vs_vdd, jitter_noise_vs_vdd, jitter_spurs_vs_vdd, jitter_pnm_vs_vdd, current_vs_vdd (PLL locked; curve: c9 min 14 max 0, 1000 MHz)
  run: pll_test_vdd_S5_20261026_155550.csv
  fixed: VCO coarse 9 min 14 max 0, f_vco 1000 MHz, output /16 (pad 62.5 MHz), f_ref 7.812 MHz, loop filter Icp 7 C1 0 C2 0 R1 7, Vctrl pad off, core in reset
  varied: PLL supply nan V (E36312A CH2); jitter on the pad over 1000-1e+07 Hz (full curve) and 1000-2e+07 Hz (Phase Noise mode)



In [12]:
# Phase noise referred to the VCO (pad + 20 log10 N) per PLL supply, one figure per setting and source: the full
# curve (our spectrum method, "L_trace") and the analyzer's Phase Noise mode ("L_pnmode").
for kind, source, name in (("L_trace", "full curve", "phase_noise"),
                           ("L_pnmode", "Phase Noise mode", "phase_noise_pnm")):
    for setting, d in ok.groupby("setting", sort=False):
        fig = go.Figure()
        for _, r in d.sort_values("vdd", ascending=False).iterrows():
            t = binned(trace_of(r, kind))
            fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines",
                            name=f"{r['vdd']:.3f} V", line=dict(width=2, color=vdd_color(r["vdd"])),
                            hovertemplate=f"{r['vdd']:.3f} V" + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
        fig.update_layout(title=f"Phase noise vs PLL supply ({source}), {setting}", height=550,
                          legend=dict(orientation="h", y=-0.2))
        fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS)
        fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
        fname = f"{name}_vs_vdd_{setting.replace(' ', '').replace(',', '_')}"
        save(fig, fname)
        print_settings(f"{fname} (PLL locked, L(f) referred to the VCO, {source})", d.iloc[0],
                       "PLL supply " + ", ".join(f"{v:.3f}" for v in sorted(d["vdd"])) + " V (one curve each)")

ValueError: cannot convert NaN to integer

In [13]:
CAPTURE = 1  # 1 = +-3 MHz at 1 kHz RBW, 2 = +-10 MHz at 3 kHz, 3 = +-1 MHz at 300 Hz (pll_spur_zoom.SHOWCASE_CAPTURES)
VDD_OVERLAY = (0.75, 0.70, 0.65, 0.60)  # [V] the supplies in the overlay figure
FLOOR_BIN, FLOOR_QUANTILE = 50e3, 0.2  # floor between the spurs: lower 20 % of the points per 50 kHz

# Spectrum around the locked carrier per PLL supply (both sidebands) in dBc (in the RBW) referred to the VCO: level
# relative to the carrier + 20 log10(N), so the spurs read their true level. The floor between the spurs is noise:
# in dBc/Hz referred to the VCO (- 10 log10(RBW) on top), the scale of the phase-noise plots. Per setting one figure
# per supply (same axes), one overlay of VDD_OVERLAY and one with the floor of every supply.
sp = spectra[spectra["capture"] == CAPTURE].copy()
sp["dbc_vco"] = sp["level_dbm"] - sp["carrier_dbm"] + 20 * np.log10(sp["total_div"])
for setting, d in sp.groupby("setting", sort=False):
    tag = setting.replace(" ", "").replace(",", "_")
    y_range = [d["dbc_vco"].quantile(0.001) - 3, d["dbc_vco"].max() + 3]  # the same axes for every supply
    fig_overlay, fig_floor = go.Figure(), go.Figure()
    for vdd, t in sorted(d.groupby("vdd_set"), key=lambda x: -x[0]):
        x = (t["freq"] - t["f_carrier"]) / 1e6
        y = t["dbc_vco"]
        fig = go.Figure()
        fig.add_scatter(x=x, y=y, mode="lines", line=dict(width=1, color="#2a78d6"), showlegend=False,
                        hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
        fig.update_layout(title=f"Spectrum around the carrier, PLL supply {vdd:.3f} V, {setting}", height=550)
        fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
        fig.update_yaxes(title=f"level referred to the VCO [dBc in {t['rbw'].iloc[0]:g} Hz RBW]", gridcolor=GRID,
                         range=y_range)
        save(fig, f"spectrum_{tag}_{vdd * 1000:.0f}mV")
        if round(vdd, 3) in VDD_OVERLAY:
            fig_overlay.add_scatter(x=x, y=y, mode="lines", name=f"{vdd:.2f} V",
                                    line=dict(width=1, color=vdd_color(vdd)),
                                    hovertemplate=f"{vdd:.3f} V" + "<br>%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
        bins = np.floor(x.to_numpy() * 1e6 / FLOOR_BIN)
        l_vco = y.to_numpy() - 10 * np.log10(t["rbw"].iloc[0])
        fig_floor.add_scatter(x=[(b + 0.5) * FLOOR_BIN / 1e6 for b in np.unique(bins)],
                              y=[np.percentile(l_vco[bins == b], 100 * FLOOR_QUANTILE) for b in np.unique(bins)],
                              mode="lines", name=f"{vdd:.3f} V", line=dict(width=2, color=vdd_color(vdd)),
                              hovertemplate=f"{vdd:.3f} V" + "<br>%{x:.3f} MHz: %{y:.1f} dBc/Hz<extra></extra>")
    fig_overlay.update_layout(title=f"Spectrum around the carrier vs PLL supply, {setting}", height=550,
                              legend=dict(orientation="h", y=-0.2))
    fig_overlay.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
    fig_overlay.update_yaxes(title=f"level referred to the VCO [dBc in {d['rbw'].iloc[0]:g} Hz RBW]",
                             gridcolor=GRID, range=y_range)
    save(fig_overlay, f"spectrum_{tag}_overlay")
    fig_floor.update_layout(title=f"Noise floor between the spurs vs PLL supply, {setting}", height=550,
                            legend=dict(orientation="h", y=-0.2))
    fig_floor.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
    fig_floor.update_yaxes(title="floor referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig_floor, f"spectrum_{tag}_floor")
    r = d.iloc[0]
    print_settings(f"spectrum_{tag}_<mV>, spectrum_{tag}_overlay, spectrum_{tag}_floor (PLL locked, pad spectrum)", r,
                   "PLL supply " + ", ".join(f"{v:.3f}" for v in sorted(d["vdd_set"].unique())) + " V; "
                   f"span +-{r['span'] / 2e6:g} MHz, RBW {r['rbw']:g} Hz, {r['averages']:.0f} averages, centred on the "
                   f"carrier; spectra in dBc in the RBW referred to the VCO (+ 20 log10({r['total_div']:.0f})); "
                   f"floor = lower {FLOOR_QUANTILE:.0%} of the points per {FLOOR_BIN / 1e3:g} kHz, in dBc/Hz referred "
                   f"to the VCO (- 10 log10({r['rbw']:g} Hz) on top)")

In [14]:
VCO_VDD_CSV = None  # e.g. PLL_DIR.parent / "vco" / "vco_vdd_S5_20261022_100000.csv"; None = newest run

# Open loop (pll_test_vdd.vco_vdd_sweep): the free-running VCO with the codes and pad divider of every setting, Vctrl
# from the SMU retuned to the setting's f_vco at every PLL supply. Per setting the analyzer's Phase Noise mode L(f)
# referred to the VCO (pad + 20 log10 N), one curve per supply, and its RMS jitter against the supply.
if VCO_VDD_CSV is None:
    VCO_VDD_CSV = sorted((p for p in (PLL_DIR.parent / "vco").glob("vco_vdd_*.csv") if not p.stem.endswith("_traces")),
                         key=lambda p: p.stat().st_mtime)[-1]
VCO_VDD_CSV = Path(VCO_VDD_CSV)
ol = pd.read_csv(VCO_VDD_CSV)
ol_traces = pd.read_csv(VCO_VDD_CSV.with_name(VCO_VDD_CSV.stem + "_traces.csv"))
ol_traces = ol_traces[ol_traces["kind"] == "L_pnmode"]
print(f"{VCO_VDD_CSV.name}: {len(ol)} supply x setting, status "
      + ", ".join(f"{s} {n}" for s, n in ol["status"].value_counts().items()))
ol_ok = ol[ol["status"] == "ok"]
for setting, d in ol_ok.groupby("factor", sort=False):
    fig = go.Figure()
    for _, r in d.sort_values("vdd_set", ascending=False).drop_duplicates("vdd_set").iterrows():
        t = ol_traces[(ol_traces["vdd_set"] == r["vdd_set"]) & (ol_traces["f_vco_target"] == r["f_vco_target"])
                      & np.logical_and.reduce([ol_traces[c] == r[c] for c in CODES])]
        t = binned(t)
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines",
                        name=f"{r['vdd_set']:.3f} V", line=dict(width=2, color=vdd_color(r["vdd_set"])),
                        hovertemplate=f"{r['vdd_set']:.3f} V" + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Open-loop phase noise vs PLL supply (Phase Noise mode), {setting}", height=550,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS)
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    tag = setting.replace(" ", "_")
    save(fig, f"open_loop_phase_noise_vs_vdd_{tag}")
    r = d.iloc[0]
    print(f"open_loop_phase_noise_vs_vdd_{tag} (open loop, L(f) referred to the VCO, Phase Noise mode)\n"
          f"  run: {VCO_VDD_CSV.name}\n"
          f"  fixed: VCO coarse {r['vco_tune_coarse']:.0f} min {r['vco_current_min']:.0f} "
          f"max {r['vco_current_max']:.0f}, f_vco {r['f_vco_target'] / 1e6:.0f} MHz (Vctrl from the SMU, retuned), "
          f"output /{r['total_div']:.0f} (pad {r['f_pad'] / 1e6:.4g} MHz), no loop, core in reset\n"
          f"  varied: PLL supply " + ", ".join(f"{v:.3f}" for v in sorted(d["vdd_set"].unique())) + " V\n")

fig = go.Figure()
for setting, d in ol_ok.groupby("factor", sort=False):
    d = d.sort_values("vdd_set").drop_duplicates("vdd_set")
    fig.add_scatter(x=d["vdd_set"], y=d["time_jitter_pnm_ps"], mode="lines+markers", name=setting,
                    marker=dict(size=9), hovertemplate="%{x:.3f} V: %{y:.1f} ps<extra>" + setting + "</extra>")
fig.update_layout(title="Open-loop RMS jitter (Phase Noise mode) vs PLL supply", height=500,
                  legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(title="PLL supply [V]", gridcolor=GRID)
fig.update_yaxes(title="RMS jitter [ps]", gridcolor=GRID, rangemode="tozero")
save(fig, "open_loop_jitter_vs_vdd")
print(f"open_loop_jitter_vs_vdd (open loop, jitter on the pad, Phase Noise mode)\n  run: {VCO_VDD_CSV.name}\n"
      f"  varied: PLL supply (x), setting (one curve each); band per setting: "
      + ", ".join(f"{s} {d['pnm_band'].iloc[0]} Hz" for s, d in ol_ok.groupby("factor", sort=False)) + "\n")

vco_vdd_S5_20261022_084755.csv: 18 supply x setting, status ok 11, no_carrier 4, unreachable 2, error 1


open_loop_phase_noise_vs_vdd_c9_min14_max0_1GHz (open loop, L(f) referred to the VCO, Phase Noise mode)
  run: vco_vdd_S5_20261022_084755.csv
  fixed: VCO coarse 9 min 14 max 0, f_vco 1000 MHz (Vctrl from the SMU, retuned), output /16 (pad 62.5 MHz), no loop, core in reset
  varied: PLL supply 0.625, 0.675, 0.700, 0.750, 0.800 V



open_loop_phase_noise_vs_vdd_c9_min9_max0_760MHz (open loop, L(f) referred to the VCO, Phase Noise mode)
  run: vco_vdd_S5_20261022_084755.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz (Vctrl from the SMU, retuned), output /8 (pad 95 MHz), no loop, core in reset
  varied: PLL supply 0.600, 0.625, 0.650, 0.675, 0.700 V



open_loop_jitter_vs_vdd (open loop, jitter on the pad, Phase Noise mode)
  run: vco_vdd_S5_20261022_084755.csv
  varied: PLL supply (x), setting (one curve each); band per setting: c9 min14 max0 1GHz 1000-2e+07 Hz, c9 min9 max0 760MHz 1000-2e+07 Hz

